# 06 — Analyse d'erreurs et recommandations

**Projet** : Estimation du prix de vente immobilier (AVM) avec TensorFlow (GradientTape)
**Principe** : une métrique globale ne dit **jamais** quoi corriger. En régression, une RMSE
correcte peut cacher une sous-estimation systématique d'un quartier entier. Ce notebook descend
au niveau de la ligne : où le modèle se trompe-t-il, dans quel sens, et que fait-on lundi matin ?

Le split de **test** n'est utilisé qu'ici — une seule fois — pour rester une estimation honnête.

## Objectifs pédagogiques

1. Produire une évaluation complète (métriques globales, résidus, couverture de fourchette).
1. Détecter un **biais segmenté** : le modèle se trompe-t-il toujours dans le même sens ?
1. Identifier les pires erreurs et leur cause probable (queue de distribution, marché fin).
1. Formuler des recommandations concrètes, appuyées sur les chiffres observés.

**Objectifs transverses du dépôt**

- Normaliser une cible continue en interne : pourquoi un MSE sur des montants en euros diverge et comment l'éviter.
- Comprendre ce que coûte le deep learning sur données tabulaires et quand lui préférer un booster.
- Régulariser un réseau : dropout, weight decay, batch norm, early stopping avec restauration du meilleur état.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=12",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 90000.0)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

## 1. Évaluation sur le split de test

In [ ]:
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"],
    PREPARED["y_test"],
    split="test",
    context=PREPARED["enriched"]["test"],
)

metrics_frame = pd.DataFrame(
    {"métrique": list(RESULT.metrics), "valeur": [RESULT.metrics[name] for name in RESULT.metrics]}
)
print(f"observations évaluées : {RESULT.n_samples}")
print(f"biais moyen           : {RESULT.bias:,.0f}")
print(f"couverture ± {EVALUATOR.tolerance_pct:.0f} %   : {RESULT.coverage:.1%}")
print(f"part hors fourchette  : {RESULT.error_rate:.1%}")
metrics_frame.round(4)

**Ce qu'il faut retenir**

- La métrique de décision est `rmse` = **celle affichée ci-dessus** — c'est elle qui pilote le seuil de qualité.
- La RMSE se lit **avec** la MAE : un rapport RMSE/MAE élevé (> 1,6) signale une queue d'erreur lourde, pas un modèle globalement mauvais.
- Le `context` passé à `evaluate()` permet d'enrichir l'analyse avec les colonnes brutes (identifiant, quartier, surface).
- La couverture de fourchette est la traduction métier de l'erreur : c'est elle qui détermine si l'estimation est publiable.

## 2. Prédit vs observé — le premier réflexe

In [ ]:
from src.visualization.plots import RegressionPlots

PLOTS = RegressionPlots(NB_PATHS.figures_dir)
for name in ("predicted_vs_actual", "residuals_vs_predicted"):
    path = getattr(PLOTS, name)(RESULT)
    if path is not None:
        display(Image(path, width=520))

**Ce qu'il faut retenir**

- Un nuage **centré sur la diagonale** sans structure = pas de biais global. Un nuage qui s'écarte de la diagonale aux extrémités = le modèle « lisse » la queue de distribution.
- Le graphique des résidus est le test d'**hétéroscédasticité** : un cône qui s'ouvre avec la valeur prédite est attendu sur un prix (erreur multiplicative).
- La ligne rouge (résidu moyen par classe) doit rester proche de zéro : une dérive monotone indique un effet non appris ou une retransformation biaisée.

## 3. Distribution de l'erreur et couverture de la fourchette

In [ ]:
for name in ("error_distribution", "error_by_bucket", "coverage_by_bucket", "error_breakdown"):
    path = getattr(PLOTS, name)(RESULT)
    if path is not None:
        display(Image(path, width=540))

In [ ]:
# Lecture chiffrée de la fourchette métier : ce que le produit publie réellement.
frame = RESULT.predictions
tolerance = EVALUATOR.tolerance_pct
print(f"fourchette publiée  : ± {tolerance:.0f} %")
print(f"couverture observée : {RESULT.coverage:.1%} (cible : >= 70 %)")
print(f"erreur médiane      : {frame['absolute_error'].median():,.0f}")
print(f"erreur P95          : {frame['absolute_error'].quantile(0.95):,.0f}")
print(f"erreur relative P95 : {frame['relative_error_pct'].abs().quantile(0.95):.2f} %")
print(f"biais relatif moyen : {frame['relative_error_pct'].mean():+.2f} %")

**Ce qu'il faut retenir**

- Une couverture sous 70 % n'est pas forcément un échec du modèle : la fourchette est peut-être **trop étroite** pour le niveau de bruit réel. Les deux leviers existent (modèle, largeur).
- L'erreur **médiane** décrit le portefeuille typique, la P95 décrit le risque : les publier ensemble évite les débats stériles sur « la » bonne métrique.
- Un biais relatif moyen non nul (> ±2 %) se corrige souvent sans ré-entraîner : correction de retransformation log, ou recalibrage par segment.

## 4. Analyse par segment — là où se décide la prochaine itération

In [ ]:
segments = RESULT.per_segment
if segments.empty:
    print("Aucun axe de segmentation exploitable sur ce run.")
else:
    display(segments.round(3).head(20))
    worst = (
        segments.assign(_abs_bias=segments["bias_pct"].abs())
        .sort_values("_abs_bias", ascending=False)
        .head(5)
    )
    print("--- segments les plus biaisés ---")
    display(worst.drop(columns=["_abs_bias"]).round(3))

**Ce qu'il faut retenir**

- Un segment dont la couverture s'effondre concentre le risque métier : c'est lui qu'il faut instrumenter en premier, pas la métrique globale.
- Vérifier ensuite le **volume** d'apprentissage du segment (`n`) : sous-représentation = sous-performance, et la réponse est donnée (feature, sur-échantillonnage, modèle dédié).
- Un biais de signe constant sur un segment est un signal fort : soit une feature manque, soit la transformation de cible n'est pas cohérente sur cette population.

## 5. Où le modèle se trompe-t-il ?

In [ ]:
if RESULT.errors.empty:
    print("Aucune erreur enregistrée sur le split de test.")
else:
    print(f"{len(RESULT.errors)} pires erreurs (triées par erreur relative décroissante)")
    display(RESULT.errors.head(12).round(2))

In [ ]:
# Segmentation des erreurs : quelle population concentre les erreurs relatives ?
frame = RESULT.predictions
segmentation_columns = [
    column
    for column in (["has_elevator", "has_outdoor_space", "energy_rating", "district"])
    if column in frame.columns
][:3]
for column in segmentation_columns:
    table = (
        frame.groupby(frame[column].astype(str))
        .agg(
            n=("y_true", "size"),
            erreur_relative_mediane=("relative_error_pct", "median"),
            biais_moyen=("residual", "mean"),
            couverture=("within_tolerance", "mean"),
        )
        .sort_values("couverture")
    )
    print(f"--- erreurs par `{column}` ---")
    display(table.round(3))

**Ce qu'il faut retenir**

- Les pires erreurs sont presque toujours les biens **atypiques** (très grande surface, marché fin, état exceptionnel) : le modèle n'a pas assez de voisins pour les situer.
- Une erreur isolée n'est pas un bug ; un **groupe** d'erreurs du même signe sur le même segment en est un (ou une feature manquante).
- Réponse possible : publier une fourchette élargie et orienter ces biens vers une revue humaine — c'est exactement ce que fait `Predictor` avec le niveau de confiance.

In [ ]:
baseline_comparison = EVALUATOR.compare_to_baseline(PREPARED["X_test"], PREPARED["y_test"])
gains = pd.DataFrame(
    {
        "modèle": ["modèle entraîné", "baseline (médiane)"],
        CONFIG.metrics.primary: [
            RESULT.metrics.get(CONFIG.metrics.primary, float("nan")),
            baseline_comparison.get(f"baseline_{CONFIG.metrics.primary}", float("nan")),
        ],
    }
)
gains.round(2)

**Ce qu'il faut retenir**

- Un modèle qui ne bat pas la baseline (médiane) n'apporte **aucune** valeur : il ne doit pas aller en production.
- En régression, comparer aussi le **R²** : il exprime directement le gain sur la variance expliquée, ce qui parle davantage aux métiers qu'une RMSE en euros.
- La baseline est recalculée sur le **même** split de test : la comparaison est loyale.

## 6. Rapport exécutable

In [ ]:
from src.evaluation.reports import ReportBuilder

REPORTER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
WRITTEN = REPORTER.build(RESULT, model=MODEL)
print(f"{len(WRITTEN)} artefacts écrits dans {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")
report_path = WRITTEN["report"]
Markdown(report_path.read_text(encoding="utf-8")[:2500] + "\n\n[…]")

**Ce qu'il faut retenir**

- Le rapport mélange **chiffres calculés** et **recommandations documentées** : il est régénérable à chaque run.
- Le même contenu est écrit en JSON (`artifacts/metrics/regression_report.json`) et en CSV (`error_by_segment.csv`, `top_errors.csv`) pour un dashboard ou une CI.

In [ ]:
recommendations = REPORTER.recommendations(RESULT)
for index, recommendation in enumerate(recommendations, start=1):
    print(f"{index:2d}. {recommendation}")

## 7. Recommandations concrètes

### issues de l'analyse (calculées ci-dessus)

Elles dépendent des métriques observées : couverture de fourchette à revoir, biais segmenté à
recalibrer, transformation de cible à corriger, segments à instrumenter.

### documentées pour ce cas d'usage

1. Publier une **fourchette** et un niveau de confiance, jamais un prix unique : c'est ce que le métier sait exploiter.
2. Piloter le modèle sur le MAPE, l'erreur médiane et la couverture de fourchette ; garder la RMSE comme indicateur de risque sur les biens chers.
3. Travailler en log-prix (transformer la cible) puis revenir en euros à l'inférence, avec correction de biais de retransformation.
4. Recalibrer par segment (quartier x tranche de surface) : un biais de +4 % en périphérie nord coûte plus qu'un biais moyen nul.
5. Surveiller la dérive du marché (taux d'intérêt, saisonnalité) : un modèle de prix immobilier se décale en quelques mois, d'où le ré-entraînement hebdomadaire.
6. Documenter et tester l'usage des proxies géographiques : mesurer l'écart d'estimation à caractéristiques égales entre quartiers est un garde-fou d'équité.
7. Ajouter des features exogènes en production (transactions voisines, taux de crédit, tension locative) : c'est le levier de performance principal au-delà de la fiche bien.

### plan d'action proposé

| Priorité | Action | Effet attendu | Comment vérifier |
| --- | --- | --- | --- |
| 1 | Recalibrer les segments biaisés (facteur multiplicatif par segment) | biais relatif < 2 % partout | §4 rejoué sur le test |
| 2 | Ajuster la largeur de fourchette au niveau de confiance | couverture ≥ 70 % | `coverage_by_bucket` |
| 3 | Ajouter les features exogènes manquantes (transactions voisines, tension du marché) | MAPE en baisse | notebook 04, comparaison d'algorithmes |
| 4 | Instrumenter la dérive du marché (prix médian observé vs estimé) | alerte précoce | `mlops/model-monitoring` |
| 5 | Rejouer ce notebook à chaque nouvelle version de données | non-régression | `make evaluate` + CI |

## 8. Limites assumées

- Les données sont **synthétiques** : les niveaux de performance illustrent une méthode, pas un marché réel.
- Une seule passe d'évaluation : la variance n'est pas mesurée ici (voir notebook 05, §4).
- L'analyse porte sur 8000 lignes générées, dont une fraction en test : les segments rares restent peu observés.
- La fourchette publiée est un intervalle **heuristique** (± pourcentage ajusté par la confiance), pas un intervalle de prédiction statistique ; un modèle quantile (ou conforme) serait nécessaire pour un niveau de confiance garanti.

**Aller plus loin dans le dépôt** : comparaison multi-stacks (`data-science/regression/with-*`),
mise en production et suivi (`mlops/`), pipelines de données (`data-eng/`).